In [1]:
import os
import glob
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ==========================
# TrustXAI Project Paths
# ==========================

PROJECT_ROOT = r"D:\TrustXAI"

RAW_DATA_PATH = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_DATA_PATH = os.path.join(PROJECT_ROOT, "data", "processed")

# Create processed folder if it does not exist
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

print("Project Root :", PROJECT_ROOT)
print("Raw Data     :", RAW_DATA_PATH)
print("Processed    :", PROCESSED_DATA_PATH)

Project Root : D:\TrustXAI
Raw Data     : D:\TrustXAI\data\raw
Processed    : D:\TrustXAI\data\processed


In [3]:
# ==========================
# Find all CSV files
# ==========================

csv_files = sorted(glob.glob(os.path.join(RAW_DATA_PATH, "*.csv")))

print(f"Total CSV files found: {len(csv_files)}")

for i, file in enumerate(csv_files, 1):
    print(f"{i}. {os.path.basename(file)}")

Total CSV files found: 8
1. Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
2. Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
3. Friday-WorkingHours-Morning.pcap_ISCX.csv
4. Monday-WorkingHours.pcap_ISCX.csv
5. Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
6. Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
7. Tuesday-WorkingHours.pcap_ISCX.csv
8. Wednesday-workingHours.pcap_ISCX.csv


In [4]:
print("PROJECT_ROOT =", PROJECT_ROOT)
print("RAW_DATA_PATH =", RAW_DATA_PATH)
print("PROCESSED_DATA_PATH =", PROCESSED_DATA_PATH)

PROJECT_ROOT = D:\TrustXAI
RAW_DATA_PATH = D:\TrustXAI\data\raw
PROCESSED_DATA_PATH = D:\TrustXAI\data\processed


In [6]:
# ==========================================
# Clean CSV files using chunks (Robust Version)
# ==========================================

CHUNK_SIZE = 100000

encodings = ["utf-8", "latin1", "cp1252"]

for i, file in enumerate(csv_files, start=1):

    print("=" * 70)
    print(f"[{i}/{len(csv_files)}] Processing: {os.path.basename(file)}")

    output_file = os.path.join(
        PROCESSED_DATA_PATH,
        os.path.basename(file)
    )

    success = False

    for enc in encodings:

        try:

            first_chunk = True
            total_before = 0
            total_after = 0

            for chunk in pd.read_csv(
                file,
                chunksize=CHUNK_SIZE,
                low_memory=False,
                encoding=enc
            ):

                total_before += len(chunk)

                # Clean column names
                chunk.columns = chunk.columns.str.strip()

                # Replace infinite values
                chunk.replace([np.inf, -np.inf], np.nan, inplace=True)

                # Remove missing values
                chunk.dropna(inplace=True)

                total_after += len(chunk)

                chunk.to_csv(
                    output_file,
                    mode="w" if first_chunk else "a",
                    header=first_chunk,
                    index=False
                )

                first_chunk = False

            print(f"Encoding Used : {enc}")
            print(f"Rows Before   : {total_before:,}")
            print(f"Rows After    : {total_after:,}")
            print(f"Removed       : {total_before-total_after:,}")
            print(f"Saved         : {output_file}")

            success = True
            break

        except UnicodeDecodeError:
            print(f"Encoding {enc} failed... trying next encoding.")

    if not success:
        print(f"Could not read: {os.path.basename(file)}")

print("\n✅ All files cleaned successfully!")

[1/8] Processing: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Encoding Used : utf-8
Rows Before   : 225,745
Rows After    : 225,711
Removed       : 34
Saved         : D:\TrustXAI\data\processed\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
[2/8] Processing: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Encoding Used : utf-8
Rows Before   : 286,467
Rows After    : 286,096
Removed       : 371
Saved         : D:\TrustXAI\data\processed\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
[3/8] Processing: Friday-WorkingHours-Morning.pcap_ISCX.csv
Encoding Used : utf-8
Rows Before   : 191,033
Rows After    : 190,911
Removed       : 122
Saved         : D:\TrustXAI\data\processed\Friday-WorkingHours-Morning.pcap_ISCX.csv
[4/8] Processing: Monday-WorkingHours.pcap_ISCX.csv
Encoding Used : utf-8
Rows Before   : 529,918
Rows After    : 529,481
Removed       : 437
Saved         : D:\TrustXAI\data\processed\Monday-WorkingHours.pcap_ISCX.csv
[5/8] Processing: Thursday-WorkingHours-A